In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import json
from typing import Any, Dict, List, Tuple, Union, Optional
import re
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier, LGBMRegressor

import lightgbm as lgb
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)

In [2]:
caminho = Path('../outputs/base_gm_tratada.parquet')
dados = pd.read_parquet(caminho)

In [3]:
# ---------------------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ---------------------------------------------------------------------------

TARGET_COL = "pontos_num"  # <-- troque pelo nome real da sua variável alvo

# Colunas categóricas (IDs e afins) -> tratadas como category nativo do LightGBM
CATEGORICAL_COLS = [
    "atleta_id",
    "clube_id",
    "adversario_id",
    "posicao_id",
    "rodada_id",
    "ano"
]

# Colunas que NÃO devem entrar no modelo (identificadores, metadados, leakage)
DROP_COLS = [
    TARGET_COL,
    'A',
    'CA',
    'CV',
    'DE',
    'DP',
    'DS',
    'FC',
    'FD',
    'FF',
    'FS',
    'FT',
    'G',
    'GC',
    'GS',
    'I',
    'PC',
    'PP',
    'PS',
    'SG',
    'entrou_em_campo',
    'equipe_visitante_id'
]

# Split temporal: tudo até (temporada, rodada) de corte vira treino;
# o restante vira validação/teste. Ajuste os cortes conforme seu histórico.
TEMPORADA_RODADA_CORTE_TREINO = (2024, 38)   # último round incluído no treino
TEMPORADA_RODADA_CORTE_VALID = (2025, 19)    # último round incluído na validação
# tudo depois disso (ou 2026 em diante) vira teste


# ---------------------------------------------------------------------------
# 2. SPLIT TEMPORAL (evita leakage de informação futura)
# ---------------------------------------------------------------------------

def split_temporal(df: pd.DataFrame,
                    corte_treino: tuple,
                    corte_valid: tuple,
                    col_temporada: str = "ano",
                    col_rodada: str = "rodada_id") -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Divide o dataframe em treino/validação/teste respeitando a ordem cronológica
    (temporada, rodada), evitando que dados futuros vazem para o treino.
    """
    chave = list(zip(df[col_temporada], df[col_rodada]))
    chave = pd.Series(chave, index=df.index)

    treino_mask = chave <= corte_treino
    valid_mask = (chave > corte_treino) & (chave <= corte_valid)
    teste_mask = chave > corte_valid

    df_treino = df.loc[treino_mask].copy()
    df_valid = df.loc[valid_mask].copy()
    df_teste = df.loc[teste_mask].copy()

    print(f"Treino:    {df_treino.shape[0]:>7} linhas | até {corte_treino}")
    print(f"Validação: {df_valid.shape[0]:>7} linhas | até {corte_valid}")
    print(f"Teste:     {df_teste.shape[0]:>7} linhas | após {corte_valid}")

    return df_treino, df_valid, df_teste


# ---------------------------------------------------------------------------
# 3. PREPARAÇÃO DE FEATURES (X, y) + tipagem categórica
# ---------------------------------------------------------------------------

def preparar_features(df: pd.DataFrame,
                       target_col: str,
                       drop_cols: list,
                       categorical_cols: list) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separa X e y, e converte as colunas categóricas para dtype 'category'
    (necessário para o LightGBM usar o tratamento nativo, sem one-hot).
    """
    y = df[target_col].copy()
    X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

    for col in categorical_cols:
        if col in X.columns:
            X[col] = X[col].astype("category")

    return X, y


# ---------------------------------------------------------------------------
# 4. TREINAMENTO COM EARLY STOPPING
# ---------------------------------------------------------------------------

def treinar_modelo(X_treino: pd.DataFrame, y_treino: pd.Series,
                    X_valid: pd.DataFrame, y_valid: pd.Series,
                    categorical_cols: list,
                    params: dict | None = None) -> lgb.LGBMRegressor:
    """
    Treina um LGBMRegressor com early stopping baseado no conjunto de validação.
    """
    default_params = dict(
        objective="regression",
        metric="rmse",
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
    )
    if params:
        default_params.update(params)

    modelo = lgb.LGBMRegressor(**default_params)

    cat_features = [c for c in categorical_cols if c in X_treino.columns]

    modelo.fit(
        X_treino, y_treino,
        eval_set=[(X_treino, y_treino), (X_valid, y_valid)],
        eval_names=["treino", "valid"],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=100),
        ],
    )

    return modelo


# ---------------------------------------------------------------------------
# 5. AVALIAÇÃO
# ---------------------------------------------------------------------------

def avaliar_modelo(modelo: lgb.LGBMRegressor,
                    X: pd.DataFrame, y: pd.Series,
                    nome_conjunto: str = "teste") -> dict:
    """
    Calcula métricas de regressão (RMSE, MAE, MAPE, R²) e imprime um resumo.
    """
    y_pred = modelo.predict(X, num_iteration=modelo.best_iteration_)

    rmse = rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    # MAPE só faz sentido se y não tiver zeros (comum em pontuação, cuidado)
    try:
        mape = mean_absolute_percentage_error(y, y_pred)
    except Exception:
        mape = np.nan

    metricas = {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2,
    }

    print(f"\n--- Métricas ({nome_conjunto}) ---")
    for nome, valor in metricas.items():
        print(f"{nome:>6}: {valor:.4f}")

    return metricas


# ---------------------------------------------------------------------------
# 6. IMPORTÂNCIA DAS FEATURES
# ---------------------------------------------------------------------------

def importancia_features(modelo: lgb.LGBMRegressor, X: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """
    Retorna um dataframe ordenado com a importância de cada feature (ganho).
    """
    importancias = pd.DataFrame({
        "feature": X.columns,
        "importancia_ganho": modelo.booster_.feature_importance(importance_type="gain"),
        "importancia_split": modelo.booster_.feature_importance(importance_type="split"),
    }).sort_values("importancia_ganho", ascending=False)

    print(f"\n--- Top {top_n} features (por ganho) ---")
    print(importancias.head(top_n).to_string(index=False))

    return importancias


# ---------------------------------------------------------------------------
# 7. PIPELINE COMPLETO
# ---------------------------------------------------------------------------

def pipeline_regressao(df: pd.DataFrame) -> dict:
    """
    Executa o pipeline completo: split -> preparação -> treino -> avaliação.
    Retorna um dicionário com o modelo, métricas e dataframes usados.
    """
    df_treino, df_valid, df_teste = split_temporal(
        df,
        corte_treino=TEMPORADA_RODADA_CORTE_TREINO,
        corte_valid=TEMPORADA_RODADA_CORTE_VALID,
    )

    X_treino, y_treino = preparar_features(df_treino, TARGET_COL, DROP_COLS, CATEGORICAL_COLS)
    X_valid, y_valid = preparar_features(df_valid, TARGET_COL, DROP_COLS, CATEGORICAL_COLS)
    X_teste, y_teste = preparar_features(df_teste, TARGET_COL, DROP_COLS, CATEGORICAL_COLS)

    modelo = treinar_modelo(X_treino, y_treino, X_valid, y_valid, CATEGORICAL_COLS)

    metricas_valid = avaliar_modelo(modelo, X_valid, y_valid, nome_conjunto="validação")
    metricas_teste = avaliar_modelo(modelo, X_teste, y_teste, nome_conjunto="teste")

    importancias = importancia_features(modelo, X_treino)

    return {
        "modelo": modelo,
        "metricas_validacao": metricas_valid,
        "metricas_teste": metricas_teste,
        "importancias": importancias,
        "X_teste": X_teste,
        "y_teste": y_teste,
    }

In [4]:
resultado = pipeline_regressao(dados)
modelo_final = resultado["modelo"]

Treino:      82927 linhas | até (2024, 38)
Validação:   13552 linhas | até (2025, 19)
Teste:       13473 linhas | após (2025, 19)
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012723 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26480
[LightGBM] [Info] Number of data points in the train set: 82927, number of used features: 207
[LightGBM] [Info] Start training from score 1.323612
Training until validation scores don't improve for 100 rounds
[100]	treino's rmse: 2.27674	valid's rmse: 2.58768
[200]	treino's rmse: 2.17809	valid's rmse: 2.58838
Early stopping, best iteration is:
[131]	treino's rmse: 2.23964	valid's rmse: 2.58463

--- Métricas (validação) ---
  RMSE: 2.5846
   MAE: 1.4624
  MAPE: 1510593948414556.0000
    R2: 0.3165

--- Métricas (teste) ---
  RMSE: 2.5891
   MAE: 1.4098
  MAPE: 1506822881102266.0000
    R2: 0.2980

--- Top 20 features (por ganho) ---
                      feature  importancia_ganho  importancia_split
          status_pre_provável  

In [5]:
# ---------------------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ---------------------------------------------------------------------------

TARGET_COL = "pontos_num"

CATEGORICAL_COLS = [
    "atleta_id",
    "clube_id",
    "adversario_id",
    "posicao_id",
    "rodada_id",
    "ano",
]

DROP_COLS = [
    TARGET_COL,
    'A',
    'CA',
    'CV',
    'DE',
    'DP',
    'DS',
    'FC',
    'FD',
    'FF',
    'FS',
    'FT',
    'G',
    'GC',
    'GS',
    'I',
    'PC',
    'PP',
    'PS',
    'SG',
    'entrou_em_campo',
    'equipe_visitante_id',
]

COL_TEMPORADA = "ano"
COL_RODADA = "rodada_id"

TEMPORADA_RODADA_CORTE_TREINO = (2024, 38)   # último round incluído no treino
TEMPORADA_RODADA_CORTE_VALID = (2025, 19)    # último round incluído na validação


# ---------------------------------------------------------------------------
# 2. SPLIT TEMPORAL (evita leakage de informação futura)
# ---------------------------------------------------------------------------

chave = pd.Series(
    list(zip(dados[COL_TEMPORADA], dados[COL_RODADA])),
    index=dados.index,
)

treino_mask = chave <= TEMPORADA_RODADA_CORTE_TREINO
valid_mask = (chave > TEMPORADA_RODADA_CORTE_TREINO) & (chave <= TEMPORADA_RODADA_CORTE_VALID)
teste_mask = chave > TEMPORADA_RODADA_CORTE_VALID

df_treino = dados.loc[treino_mask].copy()
df_valid = dados.loc[valid_mask].copy()
df_teste = dados.loc[teste_mask].copy()

print(f"Treino:    {df_treino.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_TREINO}")
print(f"Validação: {df_valid.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_VALID}")
print(f"Teste:     {df_teste.shape[0]:>7} linhas | após {TEMPORADA_RODADA_CORTE_VALID}")


# ---------------------------------------------------------------------------
# 3. PREPARAÇÃO DE FEATURES (X, y) + tipagem categórica
# ---------------------------------------------------------------------------

y_treino = df_treino[TARGET_COL].copy()
X_treino = df_treino.drop(columns=[c for c in DROP_COLS if c in df_treino.columns]).copy()

y_valid = df_valid[TARGET_COL].copy()
X_valid = df_valid.drop(columns=[c for c in DROP_COLS if c in df_valid.columns]).copy()

y_teste = df_teste[TARGET_COL].copy()
X_teste = df_teste.drop(columns=[c for c in DROP_COLS if c in df_teste.columns]).copy()

for col in CATEGORICAL_COLS:
    if col in X_treino.columns:
        X_treino[col] = X_treino[col].astype("category")
    if col in X_valid.columns:
        X_valid[col] = X_valid[col].astype("category")
    if col in X_teste.columns:
        X_teste[col] = X_teste[col].astype("category")


# ---------------------------------------------------------------------------
# 4. TREINAMENTO COM EARLY STOPPING
# ---------------------------------------------------------------------------

params = dict(
    objective="regression",
    metric="rmse",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

modelo = lgb.LGBMRegressor(**params)

cat_features = [c for c in CATEGORICAL_COLS if c in X_treino.columns]

modelo.fit(
    X_treino, y_treino,
    eval_set=[(X_treino, y_treino), (X_valid, y_valid)],
    eval_names=["treino", "valid"],
    eval_metric="rmse",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)


# ---------------------------------------------------------------------------
# 5. PREDIÇÕES (y predito) - validação e teste
# ---------------------------------------------------------------------------

y_pred_valid = modelo.predict(X_valid, num_iteration=modelo.best_iteration_)
data_predicao_valid = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

y_pred_teste = modelo.predict(X_teste, num_iteration=modelo.best_iteration_)
data_predicao_teste = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Dataframes com real vs. predito, preservando o índice original (útil para
# rastrear de volta ao atleta_id / rodada_id / ano na base original)
df_pred_valid = df_valid[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_valid["y_real"] = y_valid.values
df_pred_valid["y_predito"] = y_pred_valid
df_pred_valid["erro"] = df_pred_valid["y_real"] - df_pred_valid["y_predito"]
df_pred_valid["data_predicao"] = data_predicao_valid

df_pred_teste = df_teste[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_teste["y_real"] = y_teste.values
df_pred_teste["y_predito"] = y_pred_teste
df_pred_teste["erro"] = df_pred_teste["y_real"] - df_pred_teste["y_predito"]
df_pred_teste["data_predicao"] = data_predicao_teste

print("\n--- Amostra de predições (validação) ---")
print(df_pred_valid.head(10).to_string(index=False))

print("\n--- Amostra de predições (teste) ---")
print(df_pred_teste.head(10).to_string(index=False))


# ---------------------------------------------------------------------------
# 6. AVALIAÇÃO (métricas de desempenho)
# ---------------------------------------------------------------------------

rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

try:
    mape_valid = mean_absolute_percentage_error(y_valid, y_pred_valid)
except Exception:
    mape_valid = np.nan

metricas_valid = {"RMSE": rmse_valid, "MAE": mae_valid, "R2": r2_valid}

print("\n--- Métricas (validação) ---")
for nome, valor in metricas_valid.items():
    print(f"{nome:>6}: {valor:.4f}")

rmse_teste = np.sqrt(mean_squared_error(y_teste, y_pred_teste))
mae_teste = mean_absolute_error(y_teste, y_pred_teste)
r2_teste = r2_score(y_teste, y_pred_teste)
try:
    mape_teste = mean_absolute_percentage_error(y_teste, y_pred_teste)
except Exception:
    mape_teste = np.nan

metricas_teste = {"RMSE": rmse_teste, "MAE": mae_teste, "R2": r2_teste}

print("\n--- Métricas (teste) ---")
for nome, valor in metricas_teste.items():
    print(f"{nome:>6}: {valor:.4f}")

Treino:      82927 linhas | até (2024, 38)
Validação:   13552 linhas | até (2025, 19)
Teste:       13473 linhas | após (2025, 19)
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26480
[LightGBM] [Info] Number of data points in the train set: 82927, number of used features: 207
[LightGBM] [Info] Start training from score 1.323612
Training until validation scores don't improve for 100 rounds
[100]	treino's rmse: 2.27674	valid's rmse: 2.58768
[200]	treino's rmse: 2.17809	valid's rmse: 2.58838
Early stopping, best iteration is:
[131]	treino's rmse: 2.23964	valid's rmse: 2.58463

--- Amostra de predições (validação) ---
 ano  rodada_id  atleta_id  posicao_id  clube_id  y_real  y_predito      erro        data_predicao
2025          1      10066           4       114    1.20   2.246406 -1.046406 2026-09-10T03:14:36Z
2025          1      10082           2       101    8.50   3.464790  5.035210 2026-09-10T03:14:36Z
202

In [7]:
# ---------------------------------------------------------------------------
# 1. BASE DE OTIMIZAÇÃO (treino + validação, teste fica de fora)
# ---------------------------------------------------------------------------

df_cv = pd.concat([df_treino, df_valid], ignore_index=False).sort_values(
    [COL_TEMPORADA, COL_RODADA]
)

# Mapeia cada combinação única (ano, rodada_id) para um índice sequencial
# de "período" -> isso vira a unidade de corte temporal (não a linha individual)
periodos = (
    df_cv[[COL_TEMPORADA, COL_RODADA]]
    .drop_duplicates()
    .sort_values([COL_TEMPORADA, COL_RODADA])
    .reset_index(drop=True)
)
periodos["periodo_id"] = periodos.index

df_cv = df_cv.merge(periodos, on=[COL_TEMPORADA, COL_RODADA], how="left")

total_periodos = periodos.shape[0]
print(f"Total de períodos (ano, rodada) disponíveis para CV: {total_periodos}")


# ---------------------------------------------------------------------------
# 2. GERAÇÃO DOS FOLDS WALK-FORWARD (expanding window, por blocos de rodada)
# ---------------------------------------------------------------------------

N_SPLITS = 5              # número de folds de validação
N_RODADAS_TESTE_FOLD = 5  # tamanho da janela de teste de cada fold, em rodadas
MIN_RODADAS_TREINO = 60   # tamanho mínimo do treino no primeiro fold

folds = []
inicio_teste = MIN_RODADAS_TREINO

for _ in range(N_SPLITS):
    fim_teste = inicio_teste + N_RODADAS_TESTE_FOLD
    if fim_teste > total_periodos:
        break
    train_periodo_ids = list(range(0, inicio_teste))
    test_periodo_ids = list(range(inicio_teste, fim_teste))
    folds.append((train_periodo_ids, test_periodo_ids))
    inicio_teste = fim_teste  # expanding window: próximo treino inclui esse bloco de teste

print(f"{len(folds)} fold(s) de walk-forward gerados (de {N_SPLITS} solicitados).")
for i, (tr, te) in enumerate(folds):
    print(
        f"  Fold {i}: treino = períodos [0, {tr[-1]}] ({len(tr)} rodadas) "
        f"| teste = períodos [{te[0]}, {te[-1]}] ({len(te)} rodadas)"
    )

if len(folds) == 0:
    raise ValueError(
        "Nenhum fold foi gerado. Reduza MIN_RODADAS_TREINO, N_RODADAS_TESTE_FOLD "
        "ou N_SPLITS - não há períodos suficientes na base de CV."
    )


# ---------------------------------------------------------------------------
# 3. ESPAÇO DE BUSCA DE HIPERPARÂMETROS
# ---------------------------------------------------------------------------

param_grid = {
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth": [-1, 4, 6, 8, 10],
    "min_child_samples": [10, 20, 30, 50, 100],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0.0, 0.01, 0.1, 0.5, 1.0],
    "reg_lambda": [0.0, 0.01, 0.1, 0.5, 1.0],
}

N_ITER = 30  # nº de combinações aleatórias testadas (ajuste conforme seu orçamento de tempo)

sampler = ParameterSampler(param_grid, n_iter=N_ITER, random_state=42)


# ---------------------------------------------------------------------------
# 4. LOOP DE BUSCA: para cada combinação, treina e avalia em todos os folds
# ---------------------------------------------------------------------------

resultados = []

for idx_comb, params_candidato in enumerate(sampler):
    rmses_fold = []

    for idx_fold, (train_ids, test_ids) in enumerate(folds):
        df_fold_treino = df_cv[df_cv["periodo_id"].isin(train_ids)]
        df_fold_teste = df_cv[df_cv["periodo_id"].isin(test_ids)]

        y_fold_treino = df_fold_treino[TARGET_COL]
        X_fold_treino = df_fold_treino.drop(
            columns=[c for c in DROP_COLS if c in df_fold_treino.columns] + ["periodo_id"]
        )

        y_fold_teste = df_fold_teste[TARGET_COL]
        X_fold_teste = df_fold_teste.drop(
            columns=[c for c in DROP_COLS if c in df_fold_teste.columns] + ["periodo_id"]
        )

        for col in CATEGORICAL_COLS:
            if col in X_fold_treino.columns:
                X_fold_treino[col] = X_fold_treino[col].astype("category")
            if col in X_fold_teste.columns:
                X_fold_teste[col] = X_fold_teste[col].astype("category")

        params_fold = dict(
            objective="regression",
            metric="rmse",
            n_estimators=2000,  # teto alto; early stopping decide o real
            random_state=42,
            n_jobs=-1,
            **params_candidato,
        )

        modelo_fold = lgb.LGBMRegressor(**params_fold)
        cat_features_fold = [c for c in CATEGORICAL_COLS if c in X_fold_treino.columns]

        modelo_fold.fit(
            X_fold_treino, y_fold_treino,
            eval_set=[(X_fold_teste, y_fold_teste)],
            eval_metric="rmse",
            categorical_feature=cat_features_fold,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )

        y_pred_fold = modelo_fold.predict(X_fold_teste, num_iteration=modelo_fold.best_iteration_)
        rmse_fold = np.sqrt(mean_squared_error(y_fold_teste, y_pred_fold))
        rmses_fold.append(rmse_fold)

    rmse_medio = float(np.mean(rmses_fold))
    rmse_std = float(np.std(rmses_fold))

    resultados.append({**params_candidato, "rmse_medio": rmse_medio, "rmse_std": rmse_std})

    print(
        f"[{idx_comb + 1}/{N_ITER}] RMSE médio: {rmse_medio:.4f} (+/- {rmse_std:.4f}) "
        f"| params: {params_candidato}"
    )


# ---------------------------------------------------------------------------
# 5. RESULTADO FINAL: ranking das combinações e melhores hiperparâmetros
# ---------------------------------------------------------------------------

df_resultados = pd.DataFrame(resultados).sort_values("rmse_medio").reset_index(drop=True)

print("\n--- Top 5 combinações (menor RMSE médio primeiro) ---")
print(df_resultados.head(5).to_string(index=False))

melhores_params = (
    df_resultados.iloc[0]
    .drop(labels=["rmse_medio", "rmse_std"])
    .to_dict()
)

# Use "melhores_params" para atualizar o dict "params" do bloco de treino final,
# por exemplo:
#
# params_finais = dict(
#     objective="regression",
#     metric="rmse",
#     n_estimators=5000,
#     random_state=42,
#     n_jobs=-1,
#     **melhores_params,
# )


Total de períodos (ano, rodada) disponíveis para CV: 130
5 fold(s) de walk-forward gerados (de 5 solicitados).
  Fold 0: treino = períodos [0, 59] (60 rodadas) | teste = períodos [60, 64] (5 rodadas)
  Fold 1: treino = períodos [0, 64] (65 rodadas) | teste = períodos [65, 69] (5 rodadas)
  Fold 2: treino = períodos [0, 69] (70 rodadas) | teste = períodos [70, 74] (5 rodadas)
  Fold 3: treino = períodos [0, 74] (75 rodadas) | teste = períodos [75, 79] (5 rodadas)
  Fold 4: treino = períodos [0, 79] (80 rodadas) | teste = períodos [80, 84] (5 rodadas)
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007470 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col

/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007837 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25299
[LightGBM] [Info] Number of data points in the train set: 49464, number of used features: 207
[LightGBM] [Info] Start training from score 1.321112
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

...

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008031 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25519
[LightGBM] [Info] Number of data points in the train set: 53445, number of used features: 207
[LightGBM] [Info] Start training from score 1.314577
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25634
[LightGBM] [Info] Number of data points in the train set: 56674, number of used features: 207
[LightGBM] [Info] Start training from score 1.315161
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 25820
[LightGBM] [Info] Number of data points in the train set: 60147, number of used features: 207
[LightGBM] [Info] Start training from score 1.312444
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[30/30] RMSE médio: 2.5130 (+/- 0.0779) | params: {'subsample

In [10]:
print("\nMelhores hiperparâmetros encontrados:")
melhores_params


Melhores hiperparâmetros encontrados:


{'subsample': 0.6,
 'reg_lambda': 0.5,
 'reg_alpha': 0.5,
 'num_leaves': 15.0,
 'min_child_samples': 30.0,
 'max_depth': 4.0,
 'learning_rate': 0.05,
 'colsample_bytree': 0.6}

In [18]:
# ---------------------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ---------------------------------------------------------------------------

TARGET_COL = "pontos_num"

CATEGORICAL_COLS = [
    "atleta_id",
    "clube_id",
    "adversario_id",
    "posicao_id",
    "rodada_id",
    "ano",
]

DROP_COLS = [
    TARGET_COL,
    'A',
    'CA',
    'CV',
    'DE',
    'DP',
    'DS',
    'FC',
    'FD',
    'FF',
    'FS',
    'FT',
    'G',
    'GC',
    'GS',
    'I',
    'PC',
    'PP',
    'PS',
    'SG',
    'entrou_em_campo',
    'equipe_visitante_id',
]

COL_TEMPORADA = "ano"
COL_RODADA = "rodada_id"

TEMPORADA_RODADA_CORTE_TREINO = (2024, 38)   # último round incluído no treino
TEMPORADA_RODADA_CORTE_VALID = (2025, 19)    # último round incluído na validação


# ---------------------------------------------------------------------------
# 2. SPLIT TEMPORAL (evita leakage de informação futura)
# ---------------------------------------------------------------------------

chave = pd.Series(
    list(zip(dados[COL_TEMPORADA], dados[COL_RODADA])),
    index=dados.index,
)

treino_mask = chave <= TEMPORADA_RODADA_CORTE_TREINO
valid_mask = (chave > TEMPORADA_RODADA_CORTE_TREINO) & (chave <= TEMPORADA_RODADA_CORTE_VALID)
teste_mask = chave > TEMPORADA_RODADA_CORTE_VALID

df_treino = dados.loc[treino_mask].copy()
df_valid = dados.loc[valid_mask].copy()
df_teste = dados.loc[teste_mask].copy()

print(f"Treino:    {df_treino.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_TREINO}")
print(f"Validação: {df_valid.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_VALID}")
print(f"Teste:     {df_teste.shape[0]:>7} linhas | após {TEMPORADA_RODADA_CORTE_VALID}")


# ---------------------------------------------------------------------------
# 3. PREPARAÇÃO DE FEATURES (X, y) + tipagem categórica
# ---------------------------------------------------------------------------

y_treino = df_treino[TARGET_COL].copy()
X_treino = df_treino.drop(columns=[c for c in DROP_COLS if c in df_treino.columns]).copy()

y_valid = df_valid[TARGET_COL].copy()
X_valid = df_valid.drop(columns=[c for c in DROP_COLS if c in df_valid.columns]).copy()

y_teste = df_teste[TARGET_COL].copy()
X_teste = df_teste.drop(columns=[c for c in DROP_COLS if c in df_teste.columns]).copy()

for col in CATEGORICAL_COLS:
    if col in X_treino.columns:
        X_treino[col] = X_treino[col].astype("category")
    if col in X_valid.columns:
        X_valid[col] = X_valid[col].astype("category")
    if col in X_teste.columns:
        X_teste[col] = X_teste[col].astype("category")


# ---------------------------------------------------------------------------
# 4. TREINAMENTO COM EARLY STOPPING
# ---------------------------------------------------------------------------

melhores_params["num_leaves"] = int(melhores_params["num_leaves"])
melhores_params["max_depth"] = int(melhores_params["max_depth"])
melhores_params["min_child_samples"] = int(melhores_params["min_child_samples"])
params_finais = dict(
    objective="regression",
    metric="rmse",
    n_estimators=5000,
    random_state=42,
    n_jobs=-1,
    **melhores_params,
)

modelo = lgb.LGBMRegressor(**params_finais)

cat_features = [c for c in CATEGORICAL_COLS if c in X_treino.columns]

modelo.fit(
    X_treino, y_treino,
    eval_set=[(X_treino, y_treino), (X_valid, y_valid)],
    eval_names=["treino", "valid"],
    eval_metric="rmse",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)


# ---------------------------------------------------------------------------
# 5. PREDIÇÕES (y predito) - validação e teste
# ---------------------------------------------------------------------------

y_pred_valid = modelo.predict(X_valid, num_iteration=modelo.best_iteration_)
data_predicao_valid = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

y_pred_teste = modelo.predict(X_teste, num_iteration=modelo.best_iteration_)
data_predicao_teste = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Dataframes com real vs. predito, preservando o índice original (útil para
# rastrear de volta ao atleta_id / rodada_id / ano na base original)
df_pred_valid = df_valid[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_valid["y_real"] = y_valid.values
df_pred_valid["y_predito"] = y_pred_valid
df_pred_valid["erro"] = df_pred_valid["y_real"] - df_pred_valid["y_predito"]
df_pred_valid["data_predicao"] = data_predicao_valid

df_pred_teste = df_teste[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_teste["y_real"] = y_teste.values
df_pred_teste["y_predito"] = y_pred_teste
df_pred_teste["erro"] = df_pred_teste["y_real"] - df_pred_teste["y_predito"]
df_pred_teste["data_predicao"] = data_predicao_teste

print("\n--- Amostra de predições (validação) ---")
print(df_pred_valid.head(10).to_string(index=False))

print("\n--- Amostra de predições (teste) ---")
print(df_pred_teste.head(10).to_string(index=False))


# ---------------------------------------------------------------------------
# 6. AVALIAÇÃO (métricas de desempenho)
# ---------------------------------------------------------------------------

rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

try:
    mape_valid = mean_absolute_percentage_error(y_valid, y_pred_valid)
except Exception:
    mape_valid = np.nan

metricas_valid = {"RMSE": rmse_valid, "MAE": mae_valid, "R2": r2_valid}

print("\n--- Métricas (validação) ---")
for nome, valor in metricas_valid.items():
    print(f"{nome:>6}: {valor:.4f}")

rmse_teste = np.sqrt(mean_squared_error(y_teste, y_pred_teste))
mae_teste = mean_absolute_error(y_teste, y_pred_teste)
r2_teste = r2_score(y_teste, y_pred_teste)
try:
    mape_teste = mean_absolute_percentage_error(y_teste, y_pred_teste)
except Exception:
    mape_teste = np.nan

metricas_teste = {"RMSE": rmse_teste, "MAE": mae_teste, "R2": r2_teste}

print("\n--- Métricas (teste) ---")
for nome, valor in metricas_teste.items():
    print(f"{nome:>6}: {valor:.4f}")

Treino:      82927 linhas | até (2024, 38)
Validação:   13552 linhas | até (2025, 19)
Teste:       13473 linhas | após (2025, 19)
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010977 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26480
[LightGBM] [Info] Number of data points in the train set: 82927, number of used features: 207
[LightGBM] [Info] Start training from score 1.323612
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh